# Auditoría de Calidad de Datos — NovaMarket

**Asignatura:** Preprocesamiento de datos

**Grupo:** 05

**Integrantes:** María Andrea Torres, Ricardo Borja, Gabriel Murillo Velandia y Laura Marcela Gómez

**Fecha:** Agosto 01 de 2026


## 1. Introducción

**Objetivo.** Realizar una auditoría de calidad sobre el dataset de ventas de NovaMarket, diagnosticando su estado en las seis dimensiones de calidad de datos (completitud, exactitud, consistencia, validez, unicidad y oportunidad), cuantificando cada problema encontrado con cifras y porcentajes.

Parte del diagnóstico se apoya en una **fuente externa oficial** para la dimensión de exactitud: el catálogo de códigos postales de Colombia (Datos Abiertos / 4-72), contra el cual se contrasta la coherencia entre `codigo_postal` y `ciudad`.

## Instalación librerias

In [ ]:
!pip install pandas tabulate

## Importación librerias

In [ ]:
import pandas as pd
import unicodedata
import matplotlib.pyplot as plt
import numpy as np

from IPython.display import display

## Carga Base de Datos

In [ ]:
df = pd.read_csv("NovaMarket_datos_crudos.csv")
df.info()

### Variables
- Se identifican 14 variables agrupadas por nivel de medición de la siguiente manera:
    - Ordinales: nivel_satisfacción, codigo_postal
    - Nominales: id_pedido, id_cliente, categoria_producto, canal, producto, correo, ciudad
    - Intervalo: fecha_compra, fecha_actualizacion_stock, edad_cliente
    - Razón: unidades, precio

- Tenemos 10 variables tipo string, 3 tipo integer y 1 tipo float

In [ ]:
df.head()

In [ ]:
df.describe(include='all')

### Datos

Observamos que los datos tienen varias problemáticas como formatos de fecha distintos en la misma columna, nombres de categoria con significado similar pero escritos de diferente manera, incongruencia en datos numéricos que no deberían ser negativos.

## Diagnóstico por dimensión

### Completitud

In [ ]:
null_data = df.isnull().sum()
completeness = (1 - df.isnull().mean()) * 100
#print(100 - completeness.round(2).sort_values())

Los valores faltantes se encuentran distribuidos así:

| Columna | Faltantes | % faltante | Nivel | Razón |
|---------------------|-----------|------------|--------------|----------------------------------------------------------------------------------------------------------------------------------------------|
| correo              | 104       | 16.77%     | Bajo impacto | Es la de mayor volumen de faltantes, pero no entra en promedios, sumas ni modelos, así que su impacto operativo es bajo; puede dejarse vacía o marcarse como desconocido. |
| nivel_satisfaccion  | 46        | 7.42%      | Moderado     | Es una variable categórica susceptible a estar vacía porque no siempre se llena; su ausencia es esperable y puede imputarse como categoría "sin dato". |
| ciudad              | 31        | 5.00%      | Moderado     | Es útil para segmentación y varios cálculos, pero con solo 5% de ausencia la pérdida de representatividad es contenida. |
| precio              | 21        | 3.39%      | Moderado     | Es clave para los cálculos estadísticos; aunque la ausencia es baja, conviene imputar (p. ej. mediana) antes que eliminar registros para no sesgar la tendencia central. |

**Visualización % de valores faltantes por columna.** El gráfico ordena las columnas por proporción de faltantes, haciendo visible de un vistazo dónde se concentra el problema de completitud que la tabla anterior cuantifica.


In [ ]:
faltantes_pct = (100 - completeness).sort_values(ascending=True)
faltantes_pct = faltantes_pct[faltantes_pct > 0]  # solo columnas con faltantes

plt.figure(figsize=(9, 5))
barras = plt.barh(faltantes_pct.index, faltantes_pct.values,
                  color="indianred", edgecolor="white")
plt.bar_label(barras, fmt="%.2f%%", padding=3, fontsize=9)
plt.title("Valores faltantes por columna, NovaMarket")
plt.xlabel("% faltante")
plt.xlim(0, faltantes_pct.max() * 1.15)
plt.tight_layout()
plt.show()


### Validez

Se verifica que los valores cumplan el formato o regla esperada: los identificadores contra su patrón, y las fechas contra un formato de fecha reconocible.

In [ ]:
col = 'id_pedido'
pattern = r'^P\d{5}$'
n = len(df)

valid_mask = df[col].str.match(pattern).fillna(False)
total_validos   = valid_mask.sum()
total_invalidos = (~valid_mask).sum()

pct_falla_id = total_invalidos / n * 100

audit_id = pd.DataFrame([{
    "Columna":            "id_pedido",
    "Formatos Validos":   total_validos,
    "Formatos Invalidos": total_invalidos,
    "% Falla":            f"{pct_falla_id:.2f}%",
}])

audit_id.style.hide(axis="index")


In [ ]:
col = 'id_cliente'
pattern = r'^C\d{4}$'

valid_mask = df[col].str.match(pattern).fillna(False)
total_validos = valid_mask.sum()
total_invalidos = (~valid_mask).sum()

id_cliente_pct = total_invalidos / len(df) * 100

audit_id_cliente = pd.DataFrame([{
    "Columna":            "id_cliente",
    "Formatos Validos":   total_validos,
    "Formatos Invalidos": total_invalidos,
    "% Falla":            f"{id_cliente_pct:.2f}%",
}])

audit_id_cliente.style.hide(axis="index")

In [ ]:
fecha_corte = pd.Timestamp("2026-08-03")

s = df['fecha_compra'].astype(str)
con_barras = s.str.contains('/')
iso = pd.to_datetime(s.where(~con_barras), format="%Y-%m-%d", errors="coerce")
dmy = pd.to_datetime(s.where(con_barras), format="%d/%m/%Y", errors="coerce")
df['purchase_dates_converted'] = iso.fillna(dmy)

df['stock_update_dates_converted'] = pd.to_datetime(
    df['fecha_actualizacion_stock'], format="%Y-%m-%d", errors="coerce")

total_purchase_invalid_formats = df['purchase_dates_converted'].isna().sum()
total_purchase_valid_formats   = df['purchase_dates_converted'].notna().sum()

df['purchase_out_of_range'] = df['purchase_dates_converted'] >= fecha_corte
total_purchase_out_of_range = df['purchase_out_of_range'].sum()

total_stock_invalid_formats = df['stock_update_dates_converted'].isna().sum()
total_stock_valid_formats   = df['stock_update_dates_converted'].notna().sum()

df['stock_out_of_range'] = df['stock_update_dates_converted'] >= fecha_corte
total_stock_out_of_range = df['stock_out_of_range'].sum()

purchase_no_valid_percentage = (total_purchase_invalid_formats + total_purchase_out_of_range) / len(df) * 100
stock_no_valid_percentage    = (total_stock_invalid_formats + total_stock_out_of_range) / len(df) * 100

audit_dates = pd.DataFrame([
    {"Columna": "fecha_compra",
     "Formatos Validos": total_purchase_valid_formats,
     "Formatos Invalidos": total_purchase_invalid_formats,
     "Fuera de Rango": total_purchase_out_of_range,
     "% Falla": f"{purchase_no_valid_percentage:.2f}%"},
    {"Columna": "fecha_actualizacion_stock",
     "Formatos Validos": total_stock_valid_formats,
     "Formatos Invalidos": total_stock_invalid_formats,
     "Fuera de Rango": total_stock_out_of_range,
     "% Falla": f"{stock_no_valid_percentage:.2f}%"},
])
audit_dates.style.hide(axis="index")

**Rangos numéricos** (precio, unidades, edad)

In [ ]:
n = len(df)

reglas = {
    "precio":       lambda s: (s <= 0),                 # un precio de venta no puede ser <= 0
    "unidades":     lambda s: (s <= 0),                 # una compra debe tener al menos 1 unidad
    "edad_cliente": lambda s: (s < 0) | (s > 90),      # edad humana fuera de rango
}

filas = []
for col, es_invalido in reglas.items():
    serie = df[col]
    invalidos = es_invalido(serie).fillna(False).sum()
    nulos     = serie.isna().sum()
    validos   = n - invalidos - nulos
    filas.append({
        "Columna":   col,
        "Validos":   validos,
        "Invalidos": invalidos,
        "Nulos":     nulos,
        "% Falla":   f"{invalidos / n * 100:.2f}%",
    })

audit_rangos = pd.DataFrame(filas)
audit_rangos.style.hide(axis="index")


**Visualización dispersión y valores fuera de rango.** Para cada variable numérica con regla de dominio se muestran dos vistas complementarias. A la izquierda, un **gráfico de dispersión a escala completa**: cada registro es un punto, los que violan la regla aparecen en rojo y la línea punteada marca el umbral; así se ven los valores imposibles y los outliers extremos. A la derecha, un **boxplot con la escala ajustada** al rango de Tukey (Q1−1.5·IQR a Q3+1.5·IQR), que hace visible la caja y los bigotes —imposible de apreciar a escala completa porque los outliers la aplastan contra el cero—.


In [ ]:
# Dispersión (escala symlog: ve el grueso Y los outliers) + boxplot (escala ajustada)
reglas_viz = {
    "precio":       {"invalido": lambda s: s <= 0,              "umbral": [(0, "precio = 0")]},
    "unidades":     {"invalido": lambda s: s <= 0,              "umbral": [(0, "unidades = 0")]},
    "edad_cliente": {"invalido": lambda s: (s < 0) | (s > 90), "umbral": [(0, "edad = 0"), (90, "edad = 90")]},
}

rng = np.random.default_rng(0)
fig, axes = plt.subplots(3, 2, figsize=(14, 15))

for fila, (col, cfg) in enumerate(reglas_viz.items()):
    serie = df[col].dropna()
    mask  = cfg["invalido"](serie)               # True = fuera de rango
    n_inv = int(mask.sum())

    # umbral lineal de symlog = cerca de Tukey (mismo criterio que el boxplot)
    q1, q3 = serie.quantile(0.25), serie.quantile(0.75)
    iqr = q3 - q1
    fence = (q3 + 1.5 * iqr) if iqr > 0 else serie.abs().max()
    linthresh = max(abs(fence), 1)

    # --- Izquierda: dispersión con escala symlog ---
    ax = axes[fila, 0]
    x_jit = rng.normal(0, 0.04, size=len(serie))
    ax.scatter(x_jit[~mask.values], serie[~mask.values],
               s=12, color="steelblue", alpha=0.5, label="válido")
    ax.scatter(x_jit[mask.values], serie[mask.values],
               s=30, color="crimson", alpha=0.85, label=f"fuera de rango (n={n_inv})")
    ax.set_yscale("symlog", linthresh=linthresh)
    for y, _ in cfg["umbral"]:
        ax.axhline(y, color="red", linestyle="--", linewidth=1)
    ax.set_title(f"{col} — dispersión (escala symlog)")
    ax.set_ylabel(col)
    ax.set_xlim(-0.25, 0.25)
    ax.set_xticks([])
    ax.legend(fontsize=8, loc="upper right")

    # --- Derecha: boxplot con escala ajustada (rango de Tukey) ---
    ax2 = axes[fila, 1]
    ax2.boxplot(serie, vert=True, showfliers=False)
    if iqr > 0:
        lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        ax2.set_ylim(lo, hi)
        for y, etiqueta in cfg["umbral"]:
            if lo <= y <= hi:
                ax2.axhline(y, color="red", linestyle="--", linewidth=1)
                ax2.text(1.32, y, etiqueta, color="red", va="center", fontsize=8)
    ax2.set_title(f"{col} — boxplot (escala ajustada)")
    ax2.set_ylabel(col)
    ax2.set_xticks([])

plt.suptitle("Validez de rangos numéricos: dispersión (symlog) y boxplot (escala ajustada)",
             fontsize=13)
plt.tight_layout()
plt.show()


**Formato de correo**

In [ ]:
col = 'correo'
pattern = r'^[^@\s]+@[^@\s]+\.[^@\s]+$'

valid_mask = df[col].str.match(pattern).astype('boolean').fillna(False)
total_validos   = valid_mask.sum()
total_nulos     = df[col].isna().sum()
total_invalidos = (~valid_mask & df[col].notna()).sum()

pct_falla = total_invalidos / len(df) * 100

audit_correo = pd.DataFrame([{
    "Columna":            col,
    "Formatos Validos":   int(total_validos),
    "Formatos Invalidos": int(total_invalidos),
    "Nulos":              int(total_nulos),
    "% Falla":            f"{pct_falla:.2f}%",
}])

audit_correo.style.hide(axis="index")

### Consistencia

Se verifica que un mismo concepto esté representado de forma uniforme, sin variantes de escritura (mayúsculas, tildes o sinónimos) que fragmenten una misma categoría. La forma canónica se toma como la variante más frecuente de cada grupo.

**Canal**

In [ ]:
tabla_canal = df['canal'].value_counts(dropna=False).rename_axis('canal').reset_index(name='conteo')
tabla_canal['%'] = (tabla_canal['conteo'] / len(df) * 100).round(2).astype(str) + '%'

tabla_canal.style.hide(axis="index")

**Ciudad**

In [ ]:
equivalencias = {
    'b/quilla':    'barranquilla',
    'bogota d.c.': 'bogota',
    'bogota dc':   'bogota',
}

def normalizar(s):
    if pd.isna(s):
        return s
    s = str(s).strip().lower()
    s = ''.join(c for c in unicodedata.normalize('NFKD', s) if not unicodedata.combining(c))
    return equivalencias.get(s, s)

ciudad_norm = df['ciudad'].map(normalizar)

tabla_exactitud = (
    df.assign(ciudad_norm=ciudad_norm)
      .dropna(subset=['ciudad'])
      .groupby('ciudad_norm')
      .agg(
          variantes = ('ciudad', 'nunique'),
          registros = ('ciudad', 'size'),
          canonica  = ('ciudad', lambda x: x.mode()[0]),
          desviados = ('ciudad', lambda x: (x != x.mode()[0]).sum()),
          formas    = ('ciudad', lambda x: ', '.join(sorted(x.unique()))),
      )
      .sort_values('desviados', ascending=False)
      .reset_index()
)

tabla_exactitud['inconsistente'] = tabla_exactitud['variantes'] > 1
tabla_exactitud['%'] = (tabla_exactitud['desviados'] / tabla_exactitud['registros'] * 100).round(2).astype(str) + '%'

display(tabla_exactitud.style.hide(axis="index"))

_reg  = tabla_exactitud["registros"].sum()          # registros no nulos de la columna
_desv = tabla_exactitud["desviados"].sum()           # registros fuera de la forma canónica (datos malos)
_total = len(df)                              # total de la base de datos
_pct_bd  = _desv / _total * 100               # % sobre el total de la BD
resumen = pd.DataFrame([{
    "Columna":                 "ciudad",
    "Registros a ajustar":     int(_desv),
    "Total BD":                int(_total),
    "% del total de la BD":    f"{_pct_bd:.2f}%",
}])
display(resumen.style.hide(axis="index"))


**Categoría de producto**

In [ ]:
equiv_categoria = {
    'juguetes': 'jugueteria',   # juguetes y juguetería se tratan como la misma categoría
}

def normalizar_categoria(s):
    if pd.isna(s):
        return s
    s = str(s).strip().lower()
    s = ''.join(c for c in unicodedata.normalize('NFKD', s) if not unicodedata.combining(c))
    return equiv_categoria.get(s, s)

cat_norm = df['categoria_producto'].map(normalizar_categoria)

tabla_categoria = (
    df.assign(cat_norm=cat_norm)
      .dropna(subset=['categoria_producto'])
      .groupby('cat_norm')
      .agg(
          variantes = ('categoria_producto', 'nunique'),
          registros = ('categoria_producto', 'size'),
          canonica  = ('categoria_producto', lambda x: x.mode()[0]),
          desviados = ('categoria_producto', lambda x: (x != x.mode()[0]).sum()),
          formas    = ('categoria_producto', lambda x: ', '.join(sorted(x.unique()))),
      )
      .sort_values('desviados', ascending=False)
      .reset_index()
)

tabla_categoria['inconsistente'] = tabla_categoria['variantes'] > 1
tabla_categoria['%'] = (tabla_categoria['desviados'] / tabla_categoria['registros'] * 100).round(2).astype(str) + '%'

display(tabla_categoria.style.hide(axis="index"))

_reg  = tabla_categoria["registros"].sum()          # registros no nulos de la columna
_desv = tabla_categoria["desviados"].sum()           # registros fuera de la forma canónica (datos malos)
_total = len(df)                              # total de la base de datos
_pct_bd  = _desv / _total * 100               # % sobre el total de la BD
resumen = pd.DataFrame([{
    "Columna":                 "categoria_producto",
    "Registros a ajustar":     int(_desv),
    "Total BD":                int(_total),
    "% del total de la BD":    f"{_pct_bd:.2f}%",
}])
display(resumen.style.hide(axis="index"))


**Nivel de satisfacción**

In [ ]:
tabla_nivel = (
    df.assign(nivel_norm=df['nivel_satisfaccion'].map(normalizar))
      .dropna(subset=['nivel_satisfaccion'])
      .groupby('nivel_norm')
      .agg(
          variantes = ('nivel_satisfaccion', 'nunique'),
          registros = ('nivel_satisfaccion', 'size'),
          canonica  = ('nivel_satisfaccion', lambda x: x.mode()[0]),
          desviados = ('nivel_satisfaccion', lambda x: (x != x.mode()[0]).sum()),
          formas    = ('nivel_satisfaccion', lambda x: ', '.join(sorted(map(str, x.unique())))),
      )
      .sort_values('desviados', ascending=False)
      .reset_index()
)
tabla_nivel['inconsistente'] = tabla_nivel['variantes'] > 1
tabla_nivel['%'] = (tabla_nivel['desviados'] / tabla_nivel['registros'] * 100).round(2).astype(str) + '%'

display(tabla_nivel.style.hide(axis="index"))

_reg  = tabla_nivel["registros"].sum()          # registros no nulos de la columna
_desv = tabla_nivel["desviados"].sum()           # registros fuera de la forma canónica (datos malos)
_total = len(df)                              # total de la base de datos
_pct_bd  = _desv / _total * 100               # % sobre el total de la BD
resumen = pd.DataFrame([{
    "Columna":                 "nivel_satisfaccion",
    "Registros a ajustar":     int(_desv),
    "Total BD":                int(_total),
    "% del total de la BD":    f"{_pct_bd:.2f}%",
}])
display(resumen.style.hide(axis="index"))


**Visualización — fragmentación de categorías (antes vs. después).** Para las tres columnas de mayor impacto (ciudad, categoria_producto y nivel_satisfaccion), se contrasta el conteo con la escritura **cruda** —donde una misma clase aparece partida en varias variantes (mayúsculas, tildes, sinónimos)— frente al conteo tras **normalizar** a la forma canónica. La reducción de barras es la evidencia visual del problema de consistencia y del efecto de la corrección recomendada.


In [ ]:
nivel_norm = df['nivel_satisfaccion'].map(normalizar)

comparaciones = [
    ("ciudad",             df['ciudad'],             ciudad_norm),
    ("categoria_producto", df['categoria_producto'], cat_norm),
    ("nivel_satisfaccion", df['nivel_satisfaccion'], nivel_norm),
]

fig, axes = plt.subplots(len(comparaciones), 2, figsize=(14, 12))
for fila, (nombre, cruda, norm) in enumerate(comparaciones):
    vc_crudo = cruda.value_counts(dropna=False).head(15)
    vc_norm  = norm.value_counts(dropna=False).head(15)

    axes[fila, 0].bar(range(len(vc_crudo)), vc_crudo.values,
                      color="indianred", edgecolor="white")
    axes[fila, 0].set_xticks(range(len(vc_crudo)))
    axes[fila, 0].set_xticklabels(vc_crudo.index.astype(str), rotation=45, ha="right", fontsize=8)
    axes[fila, 0].set_title(f"{nombre} — CRUDO ({cruda.nunique()} variantes)")
    axes[fila, 0].set_ylabel("Frecuencia")

    axes[fila, 1].bar(range(len(vc_norm)), vc_norm.values,
                      color="seagreen", edgecolor="white")
    axes[fila, 1].set_xticks(range(len(vc_norm)))
    axes[fila, 1].set_xticklabels(vc_norm.index.astype(str), rotation=45, ha="right", fontsize=8)
    axes[fila, 1].set_title(f"{nombre} — NORMALIZADO ({norm.nunique()} categorías)")

plt.suptitle("Consistencia: fragmentación de categorías antes vs. después de normalizar",
             fontsize=14)
plt.tight_layout()
plt.show()


### Exactitud

Se contrasta el dato contra una fuente de verdad externa: el catálogo oficial de
municipios de Colombia (DIVIPOLA, Datos Abiertos). La columna `codigo_postal` del
archivo contiene en realidad códigos DANE de municipio (5001 Medellín, 8001
Barranquilla, 11001 Bogotá, 13001 Cartagena, 68001 Bucaramanga, 76001 Cali), por
lo que el cruce se hace contra `codigo_municipio` del catálogo. Los registros sin
ciudad declarada no se cuentan como discrepancia: son inverificables y ya están
contados en completitud.

In [ ]:
equivalencias = {
    'b/quilla':    'barranquilla',
    'bogota d.c.': 'bogota',
    'bogota dc':   'bogota',
    'bogota, d.c.':'bogota',
}

def normalizar(s):
    if pd.isna(s):
        return s
    s = str(s).strip().lower()
    s = ''.join(c for c in unicodedata.normalize('NFKD', s) if not unicodedata.combining(c))
    return equivalencias.get(s, s)

cod_a_municipio = None
try:
    url_ref = "https://www.datos.gov.co/resource/ixig-z8b5.csv?$limit=50000"
    ref = pd.read_csv(url_ref)
    ref['municipio_norm'] = ref['nombre_municipio'].map(normalizar)
    cod_a_municipio = (ref.dropna(subset=['codigo_municipio'])
                          .astype({'codigo_municipio': int})
                          .groupby('codigo_municipio')['municipio_norm'].first().to_dict())
    print("Fuente: catálogo oficial descargado de datos.gov.co")
except Exception as e:
    # Mapeo DANE de los seis municipios presentes en la base (mismo catálogo)
    cod_a_municipio = {5001: 'medellin', 8001: 'barranquilla', 11001: 'bogota',
                       13001: 'cartagena', 68001: 'bucaramanga', 76001: 'cali'}
    print(f"Sin acceso a la fuente en línea ({type(e).__name__}); se usa el mapeo DIVIPOLA embebido.")

work = df.copy()
work['ciudad_norm']    = work['ciudad'].map(normalizar)
work['ciudad_oficial'] = work['codigo_postal'].map(cod_a_municipio)

mask_discrepa = work['ciudad'].notna() & work['ciudad_oficial'].notna() & \
                (work['ciudad_norm'] != work['ciudad_oficial'])
mask_nula     = work['ciudad'].isna() & work['ciudad_oficial'].notna()

tabla_mal = (work[mask_discrepa]
             .groupby(['ciudad', 'codigo_postal', 'ciudad_oficial'])
             .size().reset_index(name='registros')
             .sort_values('registros', ascending=False))
tabla_mal['%_del_total'] = (tabla_mal['registros'] / len(df) * 100).round(2).astype(str) + '%'
display(tabla_mal.style.hide(axis="index"))

total_discrepa = int(mask_discrepa.sum())
print(f"Registros con ciudad declarada que NO corresponde al código: "
      f"{total_discrepa} de {len(df)} ({total_discrepa/len(df)*100:.2f}%)")
print(f"(Además hay {int(mask_nula.sum())} registros sin ciudad con código conocido: "
      f"inverificables, contados en completitud, y recuperables desde el código.)")

### Unicidad

Se verifica que no existan duplicados en el identificador de pedido, que debe ser único por registro.

In [ ]:
col = 'id_pedido'
n = len(df)

total_dup_filas     = df[col].duplicated().sum()              # copias que sobran
total_ids_repetidos = (df[col].value_counts() > 1).sum()      # IDs distintos repetidos

pct_dup = total_dup_filas / n * 100

audit_unicidad = pd.DataFrame([{
    "Columna":                col,
    "Registros":              n,
    "Filas Duplicadas":       total_dup_filas,
    "IDs Distintos Repetidos": total_ids_repetidos,
    "% Falla":                f"{pct_dup:.2f}%",
}])

audit_unicidad.style.hide(axis="index")


### Oportunidad

Se verifica que las fechas tengan sentido temporal: se marcan como fuera de rango las fechas posteriores a la fecha actual (fechas futuras), que no deberían existir en registros de compra o de actualización de stock.

In [ ]:
audit_temporalidad = pd.DataFrame([
    {
        "Columna":         "fecha_compra",
        "Registros":       len(df),
        "Fuera de Rango":  total_purchase_out_of_range,
        "% Falla":         f"{total_purchase_out_of_range / len(df) * 100:.2f}%",
    },
    {
        "Columna":         "fecha_actualizacion_stock",
        "Registros":       len(df),
        "Fuera de Rango":  total_stock_out_of_range,
        "% Falla":         f"{total_stock_out_of_range / len(df) * 100:.2f}%",
    },
])

audit_temporalidad.style.hide(axis="index")


## 3. Catálogo de problemas


| # | Problema                                                                              | Dimensión | Columnas afectadas | Nº registros |      % | Impacto |
|---|---------------------------------------------------------------------------------------|-----------|--------------------|-------------:|-------:|---------|
| 1 | Inconsistencia de escritura (mayúsculas/tildes/variantes) en ciudad                   | Consistencia | ciudad |          351 | 56.61% | Alto    |
| 2 | Inconsistencia de escritura en categoria_producto (incluye «juguetes» / «juguetería») | Consistencia | categoria_producto |          351 | 56.61% | Alto    |
| 3 | Inconsistencia de escritura en nivel_satisfaccion (Alto/Medio/Bajo en varios casings) | Consistencia | nivel_satisfaccion |          336 | 54.19% | Alto    |
| 4 | Fechas futuras en fecha_compra                                                        | Oportunidad | fecha_compra |          132 | 21.29% | Alto    |
| 5 | Valores faltantes en correo                                                           | Completitud | correo |          104 | 16.77% | Bajo    |
| 6 | Correos con formato inválido (sin @)                                                  | Validez | correo |           51 |  8.23% | Medio   |
| 7 | Valores faltantes en nivel_satisfaccion                                               | Completitud | nivel_satisfaccion |           46 |  7.42% | Medio   |
| 8 | Código de municipio no corresponde a la ciudad declarada (contra fuente oficial)      | Exactitud | ciudad, codigo_postal |           41 |  6.61% | Medio   |
| 9 | edad_cliente fuera de rango (negativa o >90)                                          | Validez | edad_cliente |           32 |  5.16% | Medio   |
| 10 | unidades ≤ 0 (negativas o cero)                                                       | Validez | unidades |           31 |  5.00% | Medio   |
| 11 | Valores faltantes en ciudad                                                           | Completitud | ciudad |           31 |  5.00% | Medio   |
| 12 | Formato de fecha no reconocible en fecha_compra                                       | Validez | fecha_compra |           22 |  3.55% | Medio   |
| 13 | fecha_actualizacion_stock futura                                                      | Oportunidad | fecha_actualizacion_stock |           21 |  3.39% | Bajo    |
| 14 | Valores faltantes en precio                                                           | Completitud | precio |           21 |  3.39% | Medio   |
| 15 | id_pedido duplicados                                                                  | Unicidad | id_pedido |           20 |  3.23% | Medio   |
| 16 | precio ≤ 0 (negativo)                                                                 | Validez | precio |           19 |  3.06% | Medio   |
| 17 | unidades con valores atípicos extremos (>1000, p.ej. 9999)                            | Validez | unidades |           15 |  2.42% | Medio   |
| 18 | precio con valores atípicos extremos (>100M)                                          | Validez / Exactitud | precio |            6 |  0.97% | Medio   |